# 机器学习分类算法选型对比实验

**目的**：在相同数据集与特征工程下，横向对比 **7 种主流多标签分类算法** 在「证型预测」和「中药推荐」两个任务上的表现，为模型选型提供量化依据。

**对比指标**：Hamming Loss、Jaccard、F1-Micro、F1-Macro、Macro AUC

**模型清单（7 种算法，覆盖 6 大算法族）**：
| 类别 | 模型 | 说明 |
|------|------|------|
| 线性判别 | Logistic Regression (C=0.1, balanced) | 最终选用模型 |
| 线性判别 | Logistic Regression (C=1.0, unbalanced) | 对照：无类别平衡 |
| 支持向量机 | Linear SVM (balanced) | 线性核 SVM |
| 集成学习 (Bagging) | Random Forest (balanced) | 随机森林 |
| 集成学习 (Boosting) | Gradient Boosting | 梯度提升决策树 |
| 概率生成 | Multinomial NB | 多项式朴素贝叶斯 |
| 基于实例 | KNN (k=5) | K 近邻 |

## 1. 导入依赖

In [1]:
import pandas as pd
import numpy as np
import warnings
import time
warnings.filterwarnings('ignore')

from collections import Counter
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import (
    hamming_loss, jaccard_score, f1_score,
    accuracy_score, roc_auc_score
)

# 参选模型
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier

print('依赖导入完成')

依赖导入完成


## 2. 公共数据处理函数

In [2]:
DATA_PATH = 'data_final.csv'
TOP_N_SYMPTOMS = 200

def parse_list(s):
    if pd.isna(s) or str(s).strip() == '': return []
    return [x.strip() for x in str(s).split(',') if x.strip()]

df_raw = pd.read_csv(DATA_PATH)
print(f'原始数据形状: {df_raw.shape}')

原始数据形状: (1562, 5)


## 3. 数据处理 — 证型预测任务

与 `tcm_syndrome_model.ipynb` 完全一致的特征工程。

In [3]:
TOP_K_LABELS = 10

# 证型数据
df_zx = df_raw[df_raw['证型规范'].notna() & (df_raw['证型规范'].str.strip() != '')].reset_index(drop=True)

# 频次统计
zx_ct = Counter()
for ls in df_zx['证型规范'].apply(parse_list): zx_ct.update(ls)
top_zhengxing = [z for z, _ in zx_ct.most_common(TOP_K_LABELS)]

sx_ct = Counter()
for ls in df_zx['症状规范'].apply(parse_list): sx_ct.update(ls)
top_symptoms_zx = [s for s, _ in sx_ct.most_common(TOP_N_SYMPTOMS)]

yolo_ct = Counter()
for y in df_zx['YOLO标签'].dropna():
    y = str(y).strip()
    if y: yolo_ct[y] += 1
all_yolo_zx = sorted(yolo_ct.keys())

# 过滤样本
top_set_zx = set(top_zhengxing)
df_zx['labels_filtered'] = df_zx['证型规范'].apply(lambda s: [l for l in parse_list(s) if l in top_set_zx])
df_zx = df_zx[df_zx['labels_filtered'].apply(len) > 0].reset_index(drop=True)

# 编码
mlb_sym_zx = MultiLabelBinarizer(classes=top_symptoms_zx)
X_sym_zx = mlb_sym_zx.fit_transform(
    df_zx['症状规范'].apply(lambda s: [x for x in parse_list(s) if x in set(top_symptoms_zx)])
)
mlb_yolo_zx = MultiLabelBinarizer(classes=all_yolo_zx)
X_yolo_zx = mlb_yolo_zx.fit_transform(
    df_zx['YOLO标签'].apply(lambda y: [str(y).strip()] if pd.notna(y) and str(y).strip() else [])
)
X_zx = np.hstack([X_sym_zx, X_yolo_zx]).astype(np.float32)

mlb_label_zx = MultiLabelBinarizer(classes=top_zhengxing)
Y_zx = mlb_label_zx.fit_transform(df_zx['labels_filtered'])

X_train_zx, X_test_zx, Y_train_zx, Y_test_zx = train_test_split(
    X_zx, Y_zx, test_size=0.2, random_state=42
)

print(f'[证型] 特征: {X_zx.shape}, 标签: {Y_zx.shape}')
print(f'[证型] 训练集: {X_train_zx.shape}, 测试集: {X_test_zx.shape}')
print(f'[证型] 前{TOP_K_LABELS}证型: {top_zhengxing}')

[证型] 特征: (1386, 206), 标签: (1386, 10)
[证型] 训练集: (1108, 206), 测试集: (278, 206)
[证型] 前10证型: ['痰热壅肺', '痰浊阻肺', '肺肾气虚', '肺脾气虚', '痰瘀内阻', '肺肾气阴两虚', '肺脾肾虚', '风寒袭肺', '外寒内饮', '阳虚水泛']


## 4. 数据处理 — 中药预测任务

与 `tcm_herb_model.ipynb` 完全一致的特征工程。

In [4]:
TOP_K_HERBS = 150

# 中药数据
df_herb = df_raw[df_raw['中药规范'].notna() & (df_raw['中药规范'].str.strip() != '')].reset_index(drop=True)

# 频次统计
herb_ct = Counter()
for ls in df_herb['中药规范'].apply(parse_list): herb_ct.update(ls)
top_herbs = [h for h, _ in herb_ct.most_common(TOP_K_HERBS)]

sx_ct_h = Counter()
for ls in df_herb['症状规范'].apply(parse_list): sx_ct_h.update(ls)
top_symptoms_herb = [s for s, _ in sx_ct_h.most_common(TOP_N_SYMPTOMS)]

yolo_ct_h = Counter()
for y in df_herb['YOLO标签'].dropna():
    y = str(y).strip()
    if y: yolo_ct_h[y] += 1
all_yolo_herb = sorted(yolo_ct_h.keys())

# 过滤样本
top_herb_set = set(top_herbs)
df_herb['herbs_filtered'] = df_herb['中药规范'].apply(lambda s: [h for h in parse_list(s) if h in top_herb_set])
df_herb = df_herb[df_herb['herbs_filtered'].apply(len) > 0].reset_index(drop=True)

# 编码
mlb_sym_herb = MultiLabelBinarizer(classes=top_symptoms_herb)
X_sym_herb = mlb_sym_herb.fit_transform(
    df_herb['症状规范'].apply(lambda s: [x for x in parse_list(s) if x in set(top_symptoms_herb)])
)
mlb_yolo_herb = MultiLabelBinarizer(classes=all_yolo_herb)
X_yolo_herb = mlb_yolo_herb.fit_transform(
    df_herb['YOLO标签'].apply(lambda y: [str(y).strip()] if pd.notna(y) and str(y).strip() else [])
)
X_herb = np.hstack([X_sym_herb, X_yolo_herb]).astype(np.float32)

mlb_label_herb = MultiLabelBinarizer(classes=top_herbs)
Y_herb = mlb_label_herb.fit_transform(df_herb['herbs_filtered'])

X_train_herb, X_test_herb, Y_train_herb, Y_test_herb = train_test_split(
    X_herb, Y_herb, test_size=0.2, random_state=42
)

print(f'[中药] 特征: {X_herb.shape}, 标签: {Y_herb.shape}')
print(f'[中药] 训练集: {X_train_herb.shape}, 测试集: {X_test_herb.shape}')
print(f'[中药] 前10高频中药: {top_herbs[:10]}')

[中药] 特征: (1562, 206), 标签: (1562, 150)
[中药] 训练集: (1249, 206), 测试集: (313, 206)
[中药] 前10高频中药: ['茯苓', '苦杏仁', '黄芪', '紫苏子', '甘草', '白术', '五味子', '陈皮', '麻黄', '黄芩']


## 5. 评估函数与模型定义

In [5]:
def evaluate(name, model, X_tr, Y_tr, X_te, Y_te):
    """训练模型并返回评估指标字典"""
    t0 = time.time()
    model.fit(X_tr, Y_tr)
    elapsed = time.time() - t0

    Y_pred = model.predict(X_te)
    try:
        Y_proba = model.predict_proba(X_te)
        auc = roc_auc_score(Y_te, Y_proba, average='macro')
    except Exception:
        auc = float('nan')

    return {
        '模型': name,
        '训练时间(s)': round(elapsed, 1),
        'Hamming Loss ↓': round(hamming_loss(Y_te, Y_pred), 4),
        'Jaccard ↑': round(jaccard_score(Y_te, Y_pred, average='samples', zero_division=0), 4),
        'F1-Micro ↑': round(f1_score(Y_te, Y_pred, average='micro', zero_division=0), 4),
        'F1-Macro ↑': round(f1_score(Y_te, Y_pred, average='macro', zero_division=0), 4),
        'Macro AUC ↑': round(auc, 4) if not np.isnan(auc) else None,
    }


def build_models():
    """构建 7 个待对比的模型（每次调用返回全新实例）"""
    LR_PARAMS = dict(max_iter=1000, solver='lbfgs', random_state=42)
    return [
        # ① Logistic Regression — 最终选用
        ('Logistic Regression (C=0.1, balanced)',
         OneVsRestClassifier(LogisticRegression(
             C=0.1, class_weight='balanced', **LR_PARAMS), n_jobs=-1)),

        # ② Logistic Regression — 对照组
        ('Logistic Regression (C=1.0)',
         OneVsRestClassifier(LogisticRegression(
             C=1.0, **LR_PARAMS), n_jobs=-1)),

        # ③ 支持向量机
        ('Linear SVM (balanced)',
         OneVsRestClassifier(LinearSVC(
             class_weight='balanced', max_iter=2000, random_state=42), n_jobs=-1)),

        # ④ 随机森林
        ('Random Forest (balanced)',
         OneVsRestClassifier(RandomForestClassifier(
             n_estimators=200, class_weight='balanced',
             random_state=42, n_jobs=-1))),

        # ⑤ 梯度提升决策树
        ('Gradient Boosting',
         OneVsRestClassifier(GradientBoostingClassifier(
             n_estimators=100, random_state=42))),

        # ⑥ 多项式朴素贝叶斯
        ('Multinomial NB',
         OneVsRestClassifier(MultinomialNB())),

        # ⑦ K 近邻
        ('KNN (k=5)',
         OneVsRestClassifier(KNeighborsClassifier(
             n_neighbors=5, n_jobs=-1))),
    ]


def run_comparison(task_name, models, X_tr, Y_tr, X_te, Y_te):
    """对一个任务运行全部模型并返回 DataFrame"""
    results = []
    for name, model in models:
        print(f'  训练: {name}...', end=' ', flush=True)
        try:
            r = evaluate(name, model, X_tr, Y_tr, X_te, Y_te)
            results.append(r)
            auc_str = f"{r['Macro AUC ↑']:.4f}" if r['Macro AUC ↑'] else 'N/A'
            print(f"✓  {r['训练时间(s)']}s  |  F1-Macro={r['F1-Macro ↑']}  AUC={auc_str}")
        except Exception as e:
            print(f'✗ 失败: {e}')
    return pd.DataFrame(results)


def display_results(df_res, task_name):
    """按 F1-Macro 降序展示排序表格"""
    df_show = df_res.sort_values('F1-Macro ↑', ascending=False).reset_index(drop=True)
    df_show.index = df_show.index + 1  # 排名从 1 开始
    df_show.index.name = '排名'

    print(f'\n{"=" * 60}')
    print(f'  {task_name} — 模型对比结果（按 F1-Macro 降序）')
    print(f'{"=" * 60}')
    print(df_show.to_string())
    print(f'{"=" * 60}')

    # 各指标最优
    print(f'\n  各指标最优模型:')
    for col, asc in [
        ('Hamming Loss ↓', True),
        ('Jaccard ↑',      False),
        ('F1-Micro ↑',     False),
        ('F1-Macro ↑',     False),
        ('Macro AUC ↑',    False),
    ]:
        df_valid = df_res[df_res[col].notna()]
        if len(df_valid) > 0:
            best = df_valid.sort_values(col, ascending=asc).iloc[0]
            print(f"    {col}: {best['模型']}  ({best[col]})")

    return df_show

print(f'共 {len(build_models())} 个模型待对比')

共 7 个模型待对比


## 6. 对比实验一：证型预测

In [6]:
print('═' * 60)
print('  实验一：证型预测 — 7 种算法对比')
print('═' * 60)

df_res_zx = run_comparison(
    '证型预测',
    build_models(),
    X_train_zx, Y_train_zx, X_test_zx, Y_test_zx
)

df_show_zx = display_results(df_res_zx, '表1 证型预测任务 — 各算法性能对比')

════════════════════════════════════════════════════════════
  实验一：证型预测 — 7 种算法对比
════════════════════════════════════════════════════════════
  训练: Logistic Regression (C=0.1, balanced)... ✓  2.8s  |  F1-Macro=0.2943  AUC=0.6857
  训练: Logistic Regression (C=1.0)... ✓  0.0s  |  F1-Macro=0.1262  AUC=0.6832
  训练: Linear SVM (balanced)... ✓  0.0s  |  F1-Macro=0.2766  AUC=N/A
  训练: Random Forest (balanced)... ✓  1.8s  |  F1-Macro=0.1495  AUC=0.6871
  训练: Gradient Boosting... ✓  1.9s  |  F1-Macro=0.1357  AUC=0.6797
  训练: Multinomial NB... ✓  0.0s  |  F1-Macro=0.1229  AUC=0.6591
  训练: KNN (k=5)... ✓  0.0s  |  F1-Macro=0.1203  AUC=0.6134

  表1 证型预测任务 — 各算法性能对比 — 模型对比结果（按 F1-Macro 降序）
                                       模型  训练时间(s)  Hamming Loss ↓  Jaccard ↑  F1-Micro ↑  F1-Macro ↑  Macro AUC ↑
排名                                                                                                                
1   Logistic Regression (C=0.1, balanced)      2.8          0.3129     0.2464      0

## 7. 对比实验二：中药预测

In [7]:
print('═' * 60)
print('  实验二：中药预测 — 7 种算法对比')
print('═' * 60)

df_res_herb = run_comparison(
    '中药预测',
    build_models(),
    X_train_herb, Y_train_herb, X_test_herb, Y_test_herb
)

df_show_herb = display_results(df_res_herb, '表2 中药预测任务 — 各算法性能对比')

════════════════════════════════════════════════════════════
  实验二：中药预测 — 7 种算法对比
════════════════════════════════════════════════════════════
  训练: Logistic Regression (C=0.1, balanced)... ✓  0.2s  |  F1-Macro=0.1882  AUC=0.6344
  训练: Logistic Regression (C=1.0)... ✓  0.2s  |  F1-Macro=0.0533  AUC=0.6327
  训练: Linear SVM (balanced)... ✓  0.1s  |  F1-Macro=0.1857  AUC=N/A
  训练: Random Forest (balanced)... ✓  26.6s  |  F1-Macro=0.0983  AUC=0.6166
  训练: Gradient Boosting... ✓  28.5s  |  F1-Macro=0.0679  AUC=0.6283
  训练: Multinomial NB... ✓  0.1s  |  F1-Macro=0.0625  AUC=0.5884
  训练: KNN (k=5)... ✓  0.0s  |  F1-Macro=0.0611  AUC=0.5664

  表2 中药预测任务 — 各算法性能对比 — 模型对比结果（按 F1-Macro 降序）
                                       模型  训练时间(s)  Hamming Loss ↓  Jaccard ↑  F1-Micro ↑  F1-Macro ↑  Macro AUC ↑
排名                                                                                                                
1   Logistic Regression (C=0.1, balanced)      0.2          0.3085     0.1386     

## 8. 综合分析

### 算法选型结论

通过对 7 种主流机器学习分类算法在证型预测和中药推荐两个多标签分类任务上的横向对比可知：

1. **Logistic Regression (C=0.1, class_weight=balanced)** 在两个任务上 F1-Macro、Macro AUC 等综合指标均位列最优或接近最优，同时训练速度快、模型可解释性强；
2. 相比未启用 `class_weight='balanced'` 的对照组，引入代价敏感学习能有效改善低频证型/中药的召回率；
3. 随机森林和梯度提升虽在部分指标上具有竞争力，但训练耗时显著增加，且在高维稀疏二值特征场景下收益有限；
4. 朴素贝叶斯因条件独立假设的限制，在症状间存在较强关联的中医数据上表现欠佳；
5. KNN 受维度灾难影响，在 206 维稀疏特征上距离度量退化，整体性能偏低。

综上，系统最终选用 **OneVsRest + Logistic Regression (C=0.1, class_weight='balanced')** 作为证型预测与中药推荐的统一基础分类器。